<div style="background:linear-gradient(135deg,#51247a,#7a3fa0);padding:24px 28px;border-radius:10px;margin-bottom:.8rem;border-bottom:4px solid #f0a500;"><div style="font-size:2.4rem;margin-bottom:6px;">🕸️</div><div style="color:white;font-size:1.5rem;font-weight:700;">Network Visualiser</div><div style="color:rgba(255,255,255,.82);font-size:.92rem;margin-top:4px;">Generate network graphs from structured edge-list data<br><a href="https://ladal.edu.au/net.html" target="_blank" style="color:#f0c060;font-size:.85rem;">&#x2192; Open the full tutorial</a></div></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2705; How to use this tool:</b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;line-height:2.0;"><li>Upload your <code>.xlsx</code> edge-list file to <b>MyTables</b> (Step 2).</li><li>Set the filename and layout in the configuration cell (Step 3).</li><li>Click <b>Kernel &#x2192; Restart &amp; Run All</b> to run the analysis.</li><li>Download the network image from <b>MyOutput</b> (Step 4).</li></ol></div>


<div style="background:#fff8e1;border-left:4px solid #f0a500;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#7a5000;">&#x26A1; Quick start:</b> After uploading your files and editing the configuration cell below, run all cells at once by clicking <b>Kernel &#x2192; Restart &amp; Run All</b> in the menu above. You only need to run one step manually: uploading files via the file browser panel on the left.</div>


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 1</span><br><b style="font-size:.98rem;">&#x2699;&#xFE0F; Set up the environment</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
# Setup — loads helper functions used throughout the notebook.
# This cell runs automatically when you use Kernel > Restart & Run All.

suppressPackageStartupMessages(library(IRdisplay))

# Colour-coded feedback helpers
.ok   <- function(msg) IRdisplay::display_html(paste0(
  '<div style="background:#eafaf1;border-left:4px solid #27ae60;',
  'border-radius:5px;padding:9px 14px;margin:.35rem 0;font-family:sans-serif;">',
  '&#x2705; ', msg, '</div>'))
.warn <- function(msg) IRdisplay::display_html(paste0(
  '<div style="background:#fff8e1;border-left:4px solid #f0a500;',
  'border-radius:5px;padding:9px 14px;margin:.35rem 0;font-family:sans-serif;">',
  '&#x26A0;&#xFE0F; ', msg, '</div>'))
.err  <- function(msg) IRdisplay::display_html(paste0(
  '<div style="background:#fff0f0;border-left:4px solid #e74c3c;',
  'border-radius:5px;padding:9px 14px;margin:.35rem 0;font-family:sans-serif;">',
  '&#x274C; ', msg, '</div>'))
.info <- function(msg) IRdisplay::display_html(paste0(
  '<div style="background:#f4f0f8;border-left:4px solid #51247a;',
  'border-radius:5px;padding:9px 14px;margin:.35rem 0;font-family:sans-serif;">',
  '&#x2139;&#xFE0F; ', msg, '</div>'))
.prog <- function(label, pct) IRdisplay::display_html(paste0(
  '<div style="font-family:sans-serif;font-size:.85rem;margin:.3rem 0;">',
  '<b style="color:#51247a;">', label, '</b>',
  '<div style="background:#e8e4f0;border-radius:4px;height:7px;margin-top:3px;">',
  '<div style="background:#51247a;width:', round(pct), '%;height:7px;',
  'border-radius:4px;"></div></div></div>'))

# Load plain-text files from a folder
load_texts <- function(folder = "MyTexts") {
  files <- list.files(folder, pattern = "(?i)\\.txt$",
                      full.names = TRUE, recursive = FALSE)
  if (length(files) == 0)
    stop(
    "No .txt files found in '", folder, "'\n\n",
    "Please make sure:\n",
    "  1. The folder is visible in the file browser (left panel)\n",
    "  2. You dragged .txt or .TXT files into that folder\n",
    "  3. You ran Kernel > Restart & Run All AFTER uploading")
  texts <- vapply(files, function(f)
    paste(readLines(f, warn = FALSE, encoding = "UTF-8"), collapse = " "),
    character(1))
  names(texts) <- tools::file_path_sans_ext(basename(files))
  texts
}

# Save named character vector as .txt files
save_texts <- function(texts, folder = "MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  for (nm in names(texts))
    writeLines(texts[[nm]], file.path(folder, paste0(nm, ".txt")))
}

# Save data frame as Excel
save_excel <- function(df, filename, folder = "MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  writexl::write_xlsx(as.data.frame(df), file.path(folder, filename))
}

.ok("Setup complete.")


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 2</span><br><b style="font-size:.98rem;">&#x1F4C2; Upload your edge-list Excel file to MyTables</b></div>


**Input format:** three columns — `from` (start node), `to` (end node), `weight` or `s` (edge strength; use `1` for unweighted).


<div style="background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;padding:18px 22px;margin:.6rem 0;font-family:sans-serif;"><b style="color:#51247a;font-size:.95rem;">&#x1F4C2; Upload your files to <code>MyTables</code></b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;font-size:.9rem;line-height:1.9;"><li>Open the <b>file browser panel</b> on the left of the screen (click the folder icon if it is not visible).</li><li>Double-click the <b><code>MyTables</code></b> folder.</li><li><b>Drag and drop</b> your <code>.xlsx</code> files into that folder.</li><li>Then click <b>Kernel &#x2192; Restart &amp; Run All</b> to run the notebook.</li></ol><p style="margin:.5rem 0 0 0;font-size:.82rem;color:#888;">Only <code>.xlsx</code> files are accepted.</p></div>


<div style="background:#e8f4fd;border-left:4px solid #4085C6;border-radius:4px;padding:7px 14px;margin-bottom:3px;font-family:sans-serif;font-size:.82rem;color:#2a5ea8;">&#x270F;&#xFE0F; <b>Edit the values below</b>, then run this cell (Shift+Enter or click &#x25B6; Run in the toolbar).</div>


In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────
# Edit the values below, then run this cell (Shift+Enter).

data_file   <- "edgelist.xlsx" # filename in notebooks/MyTables
graph_layout <- "fr"           # "fr" (Fruchterman-Reingold), "kk"
                               # (Kamada-Kawai), "circle", or "star"
min_weight  <- 1               # minimum edge weight to include


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 3</span><br><b style="font-size:.98rem;">&#x1F4CA; Run the analysis</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
# Network visualisation — runs automatically.
suppressPackageStartupMessages(library(readxl))
suppressPackageStartupMessages(library(igraph))
suppressPackageStartupMessages(library(ggraph))
suppressPackageStartupMessages(library(tidygraph))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(scales))

path <- file.path("MyTables", data_file)
if (!file.exists(path)) {
  .err(paste0("File not found: <b>", data_file,
              "</b>. Please upload it to the MyTables folder."))
} else {
  .prog("Loading edge list...", 20)
  edges <- readxl::read_xlsx(path)
  names(edges) <- tolower(names(edges))
  if ("s" %in% names(edges) && !"weight" %in% names(edges))
    edges <- dplyr::rename(edges, weight = s)
  if (!"weight" %in% names(edges)) edges$weight <- 1
  edges <- dplyr::filter(edges, weight >= min_weight)
  .ok(paste0("Loaded <b>", nrow(edges), " edges</b> | <b>",
             length(unique(c(edges$from, edges$to))), " nodes</b>"))
  .prog("Building and rendering graph...", 60)
  g  <- igraph::graph_from_data_frame(edges, directed = FALSE)
  tg <- tidygraph::as_tbl_graph(g) |>
    tidygraph::activate(nodes) |>
    dplyr::mutate(degree = tidygraph::centrality_degree(),
                  lsize  = scales::rescale(degree, to = c(3, 8)))
  p <- ggraph::ggraph(tg, layout = graph_layout) +
    ggraph::geom_edge_link(aes(width = weight, alpha = weight),
                           colour = "gray60", show.legend = FALSE) +
    ggraph::scale_edge_width(range = c(0.3, 2.5)) +
    ggraph::scale_edge_alpha(range = c(0.2, 0.8)) +
    ggraph::geom_node_point(aes(size = degree), colour = "#51247a",
                            alpha = 0.85, show.legend = FALSE) +
    ggraph::geom_node_label(aes(label = name, size = lsize),
                            colour = "white", fill = "#51247a",
                            fontface = "bold",
                            label.padding = unit(0.15, "lines"),
                            label.size = 0, show.legend = FALSE) +
    ggraph::theme_graph(base_family = "sans") +
    labs(title = tools::file_path_sans_ext(data_file))
  print(p)
  dir.create("MyOutput", showWarnings = FALSE, recursive = TRUE)
  ggsave("notebooks/MyOutput/network_graph.png", plot = p,
         bg = "white", width = 10, height = 8, dpi = 200)
  .ok("Saved <b>network_graph.png</b> to MyOutput.")
}


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 4</span><br><b style="font-size:.98rem;">&#x2B07;&#xFE0F; Download your results</b></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:12px 18px;margin:.6rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2B07;&#xFE0F; Download your results</b><br>Open the <b>file browser</b> on the left, double-click <b>MyOutput</b>, then <b>right-click</b> any file and choose <b>Download</b>.</div>


---

### Citation

Schweinberger, Martin. (2025). *LADAL Network Visualiser*. Brisbane: The University of Queensland. <https://ladal.edu.au/tools.html>

```bibtex
@misc{schweinberger2025network,
  author = {Schweinberger, Martin},
  title  = {LADAL Network Visualiser},
  year   = {2025},
  organization = {The University of Queensland},
  url    = {https://ladal.edu.au/tools.html}
}
```


In [ ]:
sessionInfo()